In [ ]:
#import packages and modules to start sparksession and env-var

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f
import os
from dotenv import load_dotenv

In [ ]:
#load vars

env = load_dotenv('/mnt/d/linux/project/vars/.env')

pgconn = os.getenv('POSTGRE_CONN')
pguser = os.getenv('POSTGRE_USER')
pgpwd = os.getenv('POSTGRE_PWD')
pgjar = os.getenv('POSTGRE_JAR')
pgdriver = os.getenv('PG_DRIVER')


In [ ]:
#creatins session

spark = (( 
          SparkSession
            .builder
            .appName('stock_transaction')
            .config('spark.jars', f'{pgjar}')
            .config('spark.driver.extraClassPath', f'{pgjar}')
            .config('spark.executor.extraClassPath', f'{pgjar}')
        .getOrCreate()
        ))

In [ ]:
#load data from table
#creating temporary dataframe

tmp_stocktransaction = (( 
                         spark
                            .read
                            .format('jdbc')
                            .option('driver', pgdriver)
                            .option('url', pgconn)
                            .option('user', pguser)
                            .option('password', pgpwd)
                            .option('query', 
                                    
                                    """ 
                                        select
                                        
                                            stockitemtransactionid,
                                            stockitemid,
                                            supplierid,
                                            purchaseorderid,
                                            transactiontypeid,
                                            invoiceid,
                                            customerid,
                                            quantity,
                                            lasteditedby,
                                            _airbyte_ab_id,
                                            _airbyte_stockitemtransactions_hashid,
                                            _airbyte_emitted_at,
                                            _airbyte_normalized_at
                                            
                                        from warehouse.stockitemtransactions
         
                                    """                        
                                    )                            
                         )).load()

In [ ]:
#rename columns 
renamed_frame = ((tmp_stocktransaction
                    .withColumnRenamed('stockitemtransactionid', 'stocktransactionid')
                    .withColumnRenamed('_airbyte_ab_id', 'elt_hashid')
                    .withColumnRenamed('_airbyte_stockitemtransactions_hashid', 'stocktransaction_hashid')
                    .withColumnRenamed('_airbyte_emitted_at', 'elt_start')
                    .withColumnRenamed('_airbyte_normalized_at', 'elt_end')
))
renamed_frame.show(truncate=False)


In [ ]:
#remove nulls
#drop null columns
((
    renamed_frame
        .select('*', 
                    f.coalesce('supplierid', f.lit(0))
                        .alias('supplierid_n'),
                        
                    f.coalesce('customerid', f.lit(0))
                        .alias('customerid_n'),
                        
                    f.coalesce('purchaseorderid', f.lit(0))
                        .alias('purchaseorderid_n'),
                        
                    f.coalesce('invoiceid', f.lit(0))
                        .alias('invoiceid_n'),
                )
        .drop('supplierid')
        .drop('customerid')
        .drop('purchaseorderid')
        .drop('invoiceid')
        
    .show(truncate= False)
))

In [ ]:
final_stocktransaction = (( 
                           renamed_frame
                            .select('stocktransactionid',
                                    'stockitemid',
                                    'transactiontypeid',
                                    'customerid',
                                    'invoiceid',
                                    'supplierid',
                                    'purchaseorderid',
                                    'quantity',
                                    'lasteditedby',
                                    'elt_hashid',
                                    'stocktransaction_hashid',
                                    'elt_start',
                                    'elt_end')
                        ))

In [ ]:
(( 
  final_stocktransaction
    .write
    .format('jdbc')
    .option('url', pgconn)
    .option('driver', pgdriver)
    .option('user', pguser)
    .option('password', pgpwd)
    .option('dbtable', 'warehouse.testeinsert')
    .option('mode', 'overwrite')
.save(mode= 'overwrite')
))